# DeterminAgent: screen small models on a free GPU (Phase 0.4)

This notebook runs the 0.4 screen on a free T4 GPU in Google Colab or Kaggle. It installs Ollama, pulls each model, runs the golden pilot cases through the eval runner, scores them and saves the results to download. No API key is needed and nothing is paid for.

**Steps**
1. **Colab:** Runtime → Change runtime type → **T4 GPU**. **Kaggle:** Settings → Accelerator → **GPU T4**, and Settings → **Internet on** (Kaggle needs a verified phone number for both).
2. Edit the settings cell if you want fewer models. The default set takes about 30–60 minutes.
3. Runtime → **Run all**.
4. At the end, download `determinagent-screen.zip` (Colab downloads it, Kaggle puts it under Output) and send it back.

Free sessions stop after a few hours or when idle. Results are written after each model, so a stopped session keeps what finished.

In [16]:
# Settings
BRANCH = "slm-strategy"  # repo branch with the runner; use "main" once PR #6 is merged
GOLDEN = "golden/v1/pilot.yaml"
TASKS = ["rulespeak_validation", "rulespeak_generation", "gherkin_requirements"]
LIMIT = None  # e.g. 5 for a quick smoke test of each task

# Ollama tags, smallest first. All fit a 16 GB T4 at Ollama's default 4-bit quantization.
MODELS = [
    "qwen3:0.6b",
    "smollm2:360m",
    "gemma3:270m",
    "qwen3:1.7b",
    "qwen3:4b",
    "phi4-mini",
    "qwen3:8b",
]
# More shortlist models. Check each tag on https://ollama.com/library first, then add it above:
#   gemma4 (E4B, 12B), ministral-3 (3B, 8B), granite4.1 (3B, 8B), smollm3
REMOVE_AFTER_RUN = True  # delete each model after its run, to save disk

# Session overrides (set in Colab, not in the repo).
import os
os.environ["DETERMINAGENT_TIMEOUT_S"] = "45"  # default 120
# max_tokens default 4096; reasoning_effort "none" turns Qwen3 thinking off (ignored by the other models)
os.environ["DETERMINAGENT_EXTRA_BODY"] = '{"max_tokens": 1536, "reasoning_effort": "none"}'

moved old runs aside


In [17]:
import os
import pathlib
import subprocess

IN_KAGGLE = os.path.exists("/kaggle/working")
WORK = pathlib.Path("/kaggle/working" if IN_KAGGLE else "/content")
print("Kaggle" if IN_KAGGLE else "Colab (or other)", "| work dir:", WORK)
print(
    subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True
    ).stdout
    or "No GPU found: models will run on CPU, slowly."
)

Colab (or other) | work dir: /content
name, memory.total [MiB]
Tesla T4, 15360 MiB



## 1. Install and start Ollama

In [18]:
!apt-get -qq install -y zstd > /dev/null 2>&1 || true
!curl -fsSL https://ollama.com/install.sh | sh > /tmp/ollama-install.log 2>&1 && echo "ollama installed" || tail -20 /tmp/ollama-install.log

ollama installed


In [19]:
import time
import urllib.request

log = open("/tmp/ollama.log", "w")
server = subprocess.Popen(
    ["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT, env={**os.environ, "OLLAMA_KEEP_ALIVE": "30m"}
)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=2)
        print("Ollama is up:", urllib.request.urlopen("http://localhost:11434/api/version").read().decode())
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("Ollama did not start; see /tmp/ollama.log")

Ollama is up: {"version":"0.40.17"}


## 2. Get the code

In [20]:
REPO = WORK / "determinagent"
if not REPO.exists():
    !git clone -q --branch {BRANCH} https://github.com/joel-waddell/determinagent.git {REPO}
!pip install -q -e {REPO}
import sys
sys.path.insert(0, str(REPO / "src"))  # editable install is not importable until restart
os.chdir(REPO)
!git log --oneline -1

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for determinagent (pyproject.toml) ... done
084c0e1 (HEAD -> slm-strategy, origin/slm-strategy) Add a Colab/Kaggle notebook for the 0.4 screen; harden the local-model adapter


## 3. Run the screen

Each model runs every selected case with prompt v1, two checker-feedback retries and schema-constrained output. One JSONL run file per model goes to `runs/`, which `python -m determinagent.eval score` and `compare` read.

In [21]:
import json

import yaml

from determinagent.eval import score_run
from determinagent.eval.runner import run
from determinagent.models import from_spec

cases_all = yaml.safe_load(open(GOLDEN))["cases"]
cases = [c for c in cases_all if c["task"] in TASKS]
if LIMIT:
    cases = [c for t in TASKS for c in [x for x in cases if x["task"] == t][:LIMIT]]
RUNS = REPO / "runs"
RUNS.mkdir(exist_ok=True)
summary = []


def summarize(model, recs):
    rows = []
    rep = score_run(cases, recs, system=f"ollama:{model}")
    for t in TASKS:
        m = rep.subset(task=t).metrics()
        if not m["cases"]:
            continue
        rows.append(
            {
                "model": model,
                "task": t,
                "cases": m["cases"],
                "task_success": m["task_success"],
                "first_pass": m["wellformed_first_pass"],
                "wellformed": m["wellformed_final"],
                "violations": m["violation_rate"],
                "precision": m.get("validation_precision"),
                "recall": m.get("validation_recall"),
                "p95_ms": m["latency_p95_ms"],
            }
        )
    return rows


for model in MODELS:
    out = RUNS / f"{model.replace(':', '_').replace('/', '_')}.jsonl"
    if out.exists():
        print(f"{model}: already run, skipping")
        summary += summarize(model, [json.loads(line) for line in open(out)])
        continue
    print(f"\n=== {model}: pulling")
    if subprocess.run(["ollama", "pull", model], capture_output=True, text=True).returncode != 0:
        print(f"{model}: pull failed (check the tag), skipping")
        continue
    t0 = time.time()
    recs = run(cases, from_spec(f"ollama:{model}"), retries=2)
    out.write_text("".join(json.dumps(r) + "\n" for r in recs))
    rows = summarize(model, recs)
    summary += rows
    print(f"{model}: {len(recs)} cases in {time.time() - t0:.0f}s")
    for r in rows:
        print(
            f"  {r['task']:<22} success {r['task_success']:.0%}  first-pass {r['first_pass']:.0%}  p95 {r['p95_ms'] or 0:.0f} ms"
        )
    if REMOVE_AFTER_RUN:
        subprocess.run(["ollama", "rm", model], capture_output=True)


=== qwen3:0.6b: pulling
qwen3:0.6b: 60 cases in 163s
  rulespeak_validation   success 35%  first-pass 35%  p95 2421 ms
  rulespeak_generation   success 40%  first-pass 55%  p95 5914 ms
  gherkin_requirements   success 0%  first-pass 0%  p95 3266 ms

=== smollm2:360m: pulling
smollm2:360m: 60 cases in 1466s
  rulespeak_validation   success 5%  first-pass 5%  p95 32102 ms
  rulespeak_generation   success 0%  first-pass 0%  p95 34774 ms
  gherkin_requirements   success 0%  first-pass 0%  p95 30883 ms

=== gemma3:270m: pulling
gemma3:270m: 60 cases in 531s
  rulespeak_validation   success 10%  first-pass 25%  p95 12713 ms
  rulespeak_generation   success 0%  first-pass 0%  p95 18849 ms
  gherkin_requirements   success 0%  first-pass 0%  p95 14720 ms

=== qwen3:1.7b: pulling
qwen3:1.7b: 60 cases in 268s
  rulespeak_validation   success 75%  first-pass 95%  p95 2563 ms
  rulespeak_generation   success 5%  first-pass 5%  p95 12944 ms
  gherkin_requirements   success 0%  first-pass 0%  p95 47

## 4. Results

In [22]:
import pandas as pd

df = pd.DataFrame(summary)
(RUNS / "summary.csv").write_text(df.to_csv(index=False))
pd.set_option("display.float_format", "{:.2f}".format)
df.pivot_table(index="model", columns="task", values="task_success", sort=False) if len(df) else df

task,rulespeak_validation,rulespeak_generation,gherkin_requirements
model,,,
qwen3:0.6b,0.35,0.40,0.00
smollm2:360m,0.05,0.00,0.00
gemma3:270m,0.10,0.00,0.00
qwen3:1.7b,0.75,0.05,0.00
qwen3:4b,0.75,0.70,0.00
phi4-mini,0.00,0.60,0.00
qwen3:8b,0.75,0.80,0.00


In [23]:
df

,model,task,cases,task_success,first_pass,wellformed,violations,precision,recall,p95_ms
0,qwen3:0.6b,rulespeak_validation,20,0.35,0.35,0.60,0.40,0.86,0.67,2420.87
1,qwen3:0.6b,rulespeak_generation,20,0.40,0.55,0.60,0.40,NaN,NaN,5913.79
2,qwen3:0.6b,gherkin_requirements,20,0.00,0.00,0.00,1.00,NaN,NaN,3265.91
3,smollm2:360m,rulespeak_validation,20,0.05,0.05,0.10,0.90,1.00,0.11,32102.46
4,smollm2:360m,rulespeak_generation,20,0.00,0.00,0.00,1.00,NaN,NaN,34774.31
5,smollm2:360m,gherkin_requirements,20,0.00,0.00,0.00,1.00,NaN,NaN,30883.04
6,gemma3:270m,rulespeak_validation,20,0.10,0.25,0.30,0.65,1.00,0.22,12713.08
7,gemma3:270m,rulespeak_generation,20,0.00,0.00,0.00,1.00,NaN,NaN,18848.92
8,gemma3:270m,gherkin_requirements,20,0.00,0.00,0.00,1.00,NaN,NaN,14720.39
9,qwen3:1.7b,rulespeak_validation,20,0.75,0.95,1.00,0.00,0.83,0.56,2563.16


## 5. Download

This zips `runs/` (one run file per model plus `summary.csv`). Send the zip back to have it scored and written up.

In [24]:
import shutil

zip_path = shutil.make_archive(str(WORK / "determinagent-screen"), "zip", RUNS)
print("Saved", zip_path)
if not IN_KAGGLE:
    try:
        from google.colab import files

        files.download(zip_path)
    except ImportError:
        pass
else:
    print("On Kaggle: open the Output panel (right side) to download it.")

Saved /content/determinagent-screen.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Optional: expose this Ollama server

Skip this section for the normal screen. It is for running the screen from somewhere else against this GPU. A Cloudflare quick tunnel gives a public URL with **no authentication**, so anyone with the URL can use the server while the notebook runs. Stop the cell when you're done.

From the other machine:
`OLLAMA_BASE_URL=<url>/v1 python -m determinagent.eval run golden/v1/pilot.yaml --model ollama:qwen3:0.6b --out runs/qwen3.jsonl`

In [26]:
# TEMP: zip the smoke-test runs and download them
import glob, os, shutil
dirs = sorted(glob.glob("/content/runs-old-*"))
for d in dirs:
    print(d, sorted(os.listdir(d)))
os.makedirs("/tmp/smoke", exist_ok=True)
for d in dirs:
    shutil.copytree(d, "/tmp/smoke/" + os.path.basename(d), dirs_exist_ok=True)
z = shutil.make_archive("/content/determinagent-smoke", "zip", "/tmp/smoke")
print("Saved", z, os.path.getsize(z), "bytes")
from google.colab import files
files.download(z)

/content/runs-old-1795485267 ['qwen3_0.6b.jsonl']
/content/runs-old-1795488207 ['gemma3_270m.jsonl', 'phi4-mini.jsonl', 'qwen3_0.6b.jsonl', 'qwen3_1.7b.jsonl', 'qwen3_4b.jsonl', 'smollm2_360m.jsonl']
Saved /content/determinagent-smoke.zip 14075 bytes


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>